# What Tim's missing/control return did not cover — two lists

Starts from the request sent to Tim (`AI_generated/mp2_request_for_tim.tsv`: 21,737 missing variants +
1,000 controls) and his return (`MutPred2_scores_missing_and_control.tsv`). Writes two files:

| file | starts from | rows are | score shown |
|---|---|---|---|
| `controls_not_returned_by_tim.tsv` | the 1,000 controls | (variant, isoform) pairs **in SJ's file** that Tim did not return | SJ's score |
| `missing_not_scored_by_tim_tulika_reference.tsv` | the 21,737 missing variants | (variant, isoform) pairs **in Tulika's file** that Tim did not return | Tulika's score (**reference only**) |

Both files work at the isoform level: a row appears if Tim did not return that isoform, even if he
scored other isoforms of the same variant. The column `tim_scored_no_isoform` is `True` when Tim returned
**no** scored isoform at all for the variant.

A Tim row with a blank score counts as *not returned*.

Tulika's scores are shown as a reference for what is still missing, not as scores to use.

## 1. Setup

In [1]:
from pathlib import Path

import pandas as pd

from ExTSP.config import DATA_DIR, VariantAnnotations_file, GTeX_file

DATA        = Path(str(DATA_DIR))
OUT_DIR     = DATA / "AI_generated"
REQUEST     = OUT_DIR / "mp2_request_for_tim.tsv"
TIM_RETURN  = DATA / "MutPred2_scores" / "MutPred2_scores_missing_and_control.tsv"
SJ_FILE     = DATA / "MutPred2_scores" / "MutPred2_for_filtered_variants_SJ.tsv"
TULIKA_FILE = DATA / "MutPred2_scores" / "mutpred_score_variant_transcript_tulika.tsv"

OUT_CONTROLS = OUT_DIR / "controls_not_returned_by_tim.tsv"
OUT_MISSING  = OUT_DIR / "missing_not_scored_by_tim_tulika_reference.tsv"

for f in (REQUEST, TIM_RETURN, SJ_FILE, TULIKA_FILE, VariantAnnotations_file, GTeX_file):
    print(f"{'OK ' if Path(f).exists() else 'MISSING'}  {f}")

OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/AI_generated/mp2_request_for_tim.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/MutPred2_scores/MutPred2_scores_missing_and_control.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/MutPred2_scores/MutPred2_for_filtered_variants_SJ.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/MutPred2_scores/mutpred_score_variant_transcript_tulika.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/filtered/variant_annotations.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/raw/GTeX_V8_short_read_transcripts_meanTPM_expression_across_tissue_samples_Gencodev26_without_tpm_filter.tsv


## 2. Load the request and Tim's return

Only Tim's rows with a real score are kept; blank-score rows count as not returned.

In [2]:
request = pd.read_csv(REQUEST, sep="\t")
MISSING  = set(request.loc[request["missing"], "Variant"])
CONTROLS = set(request.loc[~request["missing"], "Variant"])
info = request.set_index("Variant")[["sheets", "consequence"]]   # carried into both outputs

tim = (pd.read_csv(TIM_RETURN, sep="\t")
         .rename(columns={"ID": "Variant", "MutPred2 score": "MutPred2_tim"}))
print(f"Tim rows: {len(tim)}, of which blank score: {tim['MutPred2_tim'].isna().sum()} (dropped)")
tim = tim[tim["MutPred2_tim"].notna()]

TIM_VARIANTS = set(tim["Variant"])   # variants with >=1 scored isoform
print(f"missing requested: {len(MISSING)}   controls requested: {len(CONTROLS)}")

Tim rows: 51101, of which blank score: 2 (dropped)
missing requested: 21737   controls requested: 1000


## 3. File 1 — control isoforms in SJ's file that Tim did not return

SJ's file has no genomic variant column (it's keyed by isoform + amino-acid change), so it is mapped to
variants through `variant_annotations.tsv`, the same mapping used to pick the controls. SJ and Tim both use
GTEx (Gencode v26) isoform versions, so they are compared on the **exact** versioned ID.

In [3]:
sj = (pd.read_csv(SJ_FILE, sep="\t")
        .rename(columns={"ID": "transcript", "MutPred2": "MutPred2_SJ"}))
sj = sj[sj["MutPred2_SJ"].notna()]

ann = (pd.read_csv(VariantAnnotations_file, sep="\t", low_memory=False)
         [["ENST", "aa_sub", "Variant"]].dropna().drop_duplicates())
sj_ctrl = (ann.merge(sj, left_on=["ENST", "aa_sub"], right_on=["transcript", "Substitution"])
              .loc[lambda d: d["Variant"].isin(CONTROLS), ["Variant", "transcript", "Substitution", "MutPred2_SJ"]])

tim_pairs = set(zip(tim["Variant"], tim["transcript"]))
ctrl_gap = sj_ctrl[[p not in tim_pairs for p in zip(sj_ctrl["Variant"], sj_ctrl["transcript"])]].copy()
ctrl_gap["tim_scored_no_isoform"] = ~ctrl_gap["Variant"].isin(TIM_VARIANTS)
ctrl_gap = ctrl_gap.join(info, on="Variant").sort_values(["Variant", "transcript"])

print(f"SJ isoform scores for control variants : {len(sj_ctrl)}  ({sj_ctrl['Variant'].nunique()} variants)")
print(f"  not returned by Tim                  : {len(ctrl_gap)}  ({ctrl_gap['Variant'].nunique()} variants)")
v = ctrl_gap.drop_duplicates("Variant")
print(f"    Tim returned NO isoform            : {v['tim_scored_no_isoform'].sum()} variants "
      f"({ctrl_gap['tim_scored_no_isoform'].sum()} rows)")
print(f"    Tim returned some, missed others   : {(~v['tim_scored_no_isoform']).sum()} variants "
      f"({(~ctrl_gap['tim_scored_no_isoform']).sum()} rows)")

SJ isoform scores for control variants : 3080  (1000 variants)
  not returned by Tim                  : 354  (172 variants)
    Tim returned NO isoform            : 48 variants (160 rows)
    Tim returned some, missed others   : 124 variants (194 rows)


Controls that are in the request but have **no** SJ entry through the annotation mapping would be
invisible above, so check that none exist.

In [4]:
no_sj = CONTROLS - set(sj_ctrl["Variant"])
print(f"controls with no SJ isoform score via variant_annotations: {len(no_sj)}")

controls with no SJ isoform score via variant_annotations: 0


In [5]:
ctrl_gap.to_csv(OUT_CONTROLS, sep="\t", index=False)
print(f"wrote {len(ctrl_gap)} rows -> {OUT_CONTROLS}")
ctrl_gap.head(15)

wrote 354 rows -> /workspaces/ExTSP-default-claude/src/ExTSP/data/AI_generated/controls_not_returned_by_tim.tsv


,Variant,transcript,Substitution,MutPred2_SJ,tim_scored_no_isoform,sheets,consequence
75522,10_100529817_T/C,ENST00000527595.5,Q12R,0.048,False,nonTarget,missense
75523,10_100529817_T/C,ENST00000528174.5,Q12R,0.057,False,nonTarget,missense
75524,10_100529817_T/C,ENST00000528425.5,Q12R,0.057,False,nonTarget,missense
75525,10_100529817_T/C,ENST00000529568.5,Q12R,0.065,False,nonTarget,missense
75526,10_100529817_T/C,ENST00000533549.1,Q12R,0.057,False,nonTarget,missense
75527,10_100529817_T/C,ENST00000557395.5,Q12R,0.057,False,nonTarget,missense
75869,10_110784342_G/A,ENST00000369519.3,A447T,0.180,True,CM,missense
75881,10_110784432_G/T,ENST00000369519.3,D477Y,0.449,True,CM,missense
76278,10_113159983_G/A,ENST00000470254.1,G92D,0.228,False,ASD,no_name
70493,10_20896960_C/T,ENST00000377122.8,D51N,0.181,True,CM,missense


## 4. File 2 — missing variants: isoforms Tulika scored that Tim did not return

**Duplicates.** Tulika's file repeats many rows exactly; those are collapsed. Where the same
(variant, isoform) appears with two *different* scores, both rows are kept and flagged
`tulika_conflicting_scores`.

**Isoform matching.** Tulika's isoform version numbers often differ from the GTEx versions Tim used
(e.g. `ENST00000369519.4` vs `.5`), even though it is the same isoform. Matching on the exact version
would flag thousands of isoforms as gaps that Tim actually scored, so the match is made on the isoform ID
**without the version suffix**. Both versions are kept in the output, and `isoform_in_gtex` says whether
the ID exists in the GTEx expression matrix at all. An isoform missing from GTEx could not be used in the
GTEx analysis even if it were scored.

In [6]:
tu = pd.read_csv(TULIKA_FILE, sep="\t")
tu = tu[tu["mutation"].isin(MISSING) & tu["mutpred"].notna()]
n_before = len(tu)
tu = (tu.drop_duplicates()
        .rename(columns={"mutation": "Variant", "external_gene_name": "gene",
                         "Feature": "transcript_tulika", "mutpred": "MutPred2_tulika_reference"}))
tu["isoform_id"] = tu["transcript_tulika"].str.split(".").str[0]
print(f"Tulika scored rows for missing variants: {n_before}, after dropping exact duplicates: {len(tu)}")

conflict = tu.groupby(["Variant", "transcript_tulika"])["MutPred2_tulika_reference"].transform("nunique") > 1
tu["tulika_conflicting_scores"] = conflict
print(f"(variant, isoform) pairs with two different Tulika scores: "
      f"{tu.loc[conflict, ['Variant', 'transcript_tulika']].drop_duplicates().shape[0]}")

Tulika scored rows for missing variants: 98620, after dropping exact duplicates: 49310
(variant, isoform) pairs with two different Tulika scores: 7


In [7]:
gtex_ids = set(pd.read_csv(GTeX_file, sep="\t", usecols=["transcript_id"])["transcript_id"].str.split(".").str[0])

tim_ids = set(zip(tim["Variant"], tim["transcript"].str.split(".").str[0]))
miss_gap = tu[[p not in tim_ids for p in zip(tu["Variant"], tu["isoform_id"])]].copy()
miss_gap["tim_scored_no_isoform"] = ~miss_gap["Variant"].isin(TIM_VARIANTS)
miss_gap["isoform_in_gtex"] = miss_gap["isoform_id"].isin(gtex_ids)
miss_gap = (miss_gap.join(info, on="Variant")
              [["Variant", "gene", "transcript_tulika", "Substitution", "MutPred2_tulika_reference",
                "tim_scored_no_isoform", "isoform_in_gtex", "tulika_conflicting_scores", "sheets", "consequence"]]
              .sort_values(["Variant", "transcript_tulika"]))

print(f"missing variants Tulika scored            : {tu['Variant'].nunique()}")
print(f"  with >=1 isoform Tim did not return     : {miss_gap['Variant'].nunique()} variants, {len(miss_gap)} rows")
v = miss_gap.drop_duplicates("Variant")
print(f"    Tim returned NO isoform               : {v['tim_scored_no_isoform'].sum()} variants "
      f"({miss_gap['tim_scored_no_isoform'].sum()} rows)")
print(f"    Tim returned some, missed others      : {(~v['tim_scored_no_isoform']).sum()} variants "
      f"({(~miss_gap['tim_scored_no_isoform']).sum()} rows)")
print()
print("rows by whether the isoform exists in GTEx:")
print(miss_gap.groupby(["tim_scored_no_isoform", "isoform_in_gtex"]).size().rename("rows").to_string())

missing variants Tulika scored            : 12219
  with >=1 isoform Tim did not return     : 5119 variants, 20968 rows
    Tim returned NO isoform               : 386 variants (1488 rows)
    Tim returned some, missed others      : 4733 variants (19480 rows)

rows by whether the isoform exists in GTEx:
tim_scored_no_isoform  isoform_in_gtex
False                  False                495
                       True               18985
True                   False                 19
                       True                1469


In [8]:
miss_gap.to_csv(OUT_MISSING, sep="\t", index=False)
print(f"wrote {len(miss_gap)} rows -> {OUT_MISSING}")
miss_gap.head(15)

wrote 20968 rows -> /workspaces/ExTSP-default-claude/src/ExTSP/data/AI_generated/missing_not_scored_by_tim_tulika_reference.tsv


,Variant,gene,transcript_tulika,Substitution,MutPred2_tulika_reference,tim_scored_no_isoform,isoform_in_gtex,tulika_conflicting_scores,sheets,consequence
227327,10_110820072_G/A,RBM20,ENST00000369519.4,A851T,0.034,True,True,False,CM,missense
250275,10_111011651_C/T,SHOC2,ENST00000685059.1,P528S,0.652,False,False,False,CM,missense
250176,10_111011651_C/T,SHOC2,ENST00000688928.1,P528S,0.652,False,False,False,CM,missense
250500,10_111011651_C/T,SHOC2,ENST00000689118.1,P528S,0.652,False,False,False,CM,missense
250421,10_111011651_C/T,SHOC2,ENST00000689300.1,P528S,0.652,False,False,False,CM,missense
250168,10_111011651_C/T,SHOC2,ENST00000689997.1,P167S,0.086,False,False,False,CM,missense
250175,10_111011651_C/T,SHOC2,ENST00000691369.1,P528S,0.652,False,False,False,CM,missense
250099,10_111011651_C/T,SHOC2,ENST00000691441.1,P528S,0.652,False,False,False,CM,missense
177261,10_119651854_A/G,BAG3,ENST00000369085.8,K60R,0.142,True,True,False,CM,missense
177262,10_119669851_G/A,BAG3,ENST00000369085.8,E61K,0.149,True,True,False,CM,missense


## 5. Summary

In [9]:
for name, df in [("controls  (SJ scores)", ctrl_gap), ("missing   (Tulika reference scores)", miss_gap)]:
    v = df.drop_duplicates("Variant")
    print(f"{name}: {len(df)} rows, {len(v)} variants "
          f"-> {v['tim_scored_no_isoform'].sum()} with no Tim isoform at all, "
          f"{(~v['tim_scored_no_isoform']).sum()} partially returned")

controls  (SJ scores): 354 rows, 172 variants -> 48 with no Tim isoform at all, 124 partially returned
missing   (Tulika reference scores): 20968 rows, 5119 variants -> 386 with no Tim isoform at all, 4733 partially returned
